# Highest Reviewed Restaurants in New Brunswick

This notebook identifies the most-reviewed burger restaurants in the New Brunswick, NJ dataset and visualizes their review counts.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("data/New Brunswick, NJ_Burger.csv")
PLOT_PATH = Path("plots/New_Brunswick_Highest_Reviewed_Restaurants.png")
TOP_N = 30

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

In [ ]:
restaurants = (
    pd.read_csv(DATA_PATH)
    .rename(columns={"Unnamed: 0": "Index"})
    .drop(columns=["Index", "Business ID"], errors="ignore")
)

numeric_columns = ["Latitude", "Longitude", "Reviews", "Rating"]
restaurants[numeric_columns] = restaurants[numeric_columns].apply(pd.to_numeric, errors="coerce")

top_restaurants = (
    restaurants
    .dropna(subset=["Business", "Reviews"])
    .sort_values("Reviews", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

top_restaurants.head(10)

In [ ]:
summary_columns = ["Business", "City", "Category", "Price", "Reviews", "Rating"]
top_restaurants.loc[:, summary_columns]

In [ ]:
highest_reviewed = top_restaurants.iloc[0]
print(f"Highest reviewed restaurant: {highest_reviewed['Business']} ({highest_reviewed['Reviews']:.0f} reviews, {highest_reviewed['Rating']:.1f} stars)")

## Review Count Visualization

In [ ]:
plot_data = top_restaurants.sort_values("Reviews", ascending=True)

fig, ax = plt.subplots(figsize=(12, 9))
bars = ax.barh(plot_data["Business"], plot_data["Reviews"], color="#f28e2b", alpha=0.85)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 2, bar.get_y() + bar.get_height() / 2, f"{width:.0f}", va="center", fontsize=9)

ax.set_title("Highest Reviewed Burger Restaurants in New Brunswick, NJ", fontsize=16, weight="bold", pad=16)
ax.set_xlabel("Number of Yelp Reviews", fontsize=12)
ax.set_ylabel("")
ax.set_xlim(0, plot_data["Reviews"].max() * 1.15)

fig.tight_layout()
PLOT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()

## Optional Restaurant Map

In [ ]:
try:
    import gmaps
    from config import gkey

    gmaps.configure(api_key=gkey)
    locations = top_restaurants[["Latitude", "Longitude"]].dropna()
    weights = top_restaurants.loc[locations.index, "Reviews"]

    fig = gmaps.figure(center=(40.500, -74.447), zoom_level=13)
    fig.add_layer(gmaps.heatmap_layer(locations, weights=weights, point_radius=35, max_intensity=weights.max()))
    fig
except Exception as error:
    print(f"Map skipped: {error}")